In [ ]:
print("Hello World 123")


Hello World 123


In [2]:
import sys
print(sys.executable)

/workspaces/llm-zoomcamp-firstdraft/.venv/bin/python


In [3]:
from dotenv import load_dotenv
load_dotenv()

from openai import OpenAI
import os
open_client = OpenAI(
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url="https://openrouter.ai/api/v1"
    )

In [3]:
def llm(prompt):
    response = open_client.responses.create(
        #model="google/gemini-2.5-flash",
        model="google/gemma-4-31b-it:free",
        input=prompt,
        max_output_tokens=128
    )
    return response.output_text

question = "I just discovered the course can I join now?"
answer = llm(question)
print(answer)

#print(llm("Hey, tell me a joke."))

Since I am an AI, I don’t know which specific course you are referring to! However, I can help you figure out how to find the answer.

**To get the correct information, please check these three things:**

1.  **The Enrollment Page:** Look for a "Join Now," "Enroll," or "Register" button. If the button is active, you can likely still join.
2.  **The Course Syllabus or FAQ:** Look for a section on **"Enrollment Deadlines"** or **"Late Registration."** Many online courses (like Coursera or Udemy) are "self-paced," meaning you can join at any time. Live cohorts, however, usually have a strict start date.
3.  **Contact the Instructor/Administrator:** If you aren't sure, the best move is to send a quick email.

**If you want me to help you draft an email to the instructor to ask for a late entry, just tell me:**
*   The name of the course.
*   Why you are joining late (optional, but helpful).
*   How enthusiastic you are about the material!

**If this is a course offered through a specific p

In [4]:
context = """
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we're still accepting submissions.

Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

What is the video/zoom link to the stream for the "Office Hours" or live/workshop sessions?
The zoom link is only published to instructors/presenters/TAs. Students participate via YouTube Live and submit questions to Slido.

Cloud alternatives with GPU
Check the quota and reset cycle carefully. Potential options include Google Colab, Kaggle, Databricks.
"""

prompt = f"""
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."

Question:
{question}

Context:
{context}
"""
print(prompt)
answer = llm(prompt)
print(answer)



Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."

Question:
I just discovered the course can I join now?

Context:

I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we're still accepting submissions.

Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

What is the video/zoom link to the stream for the "Office Hours" or live/workshop sessions?
The zoom link is only published to instructors/presenters/TAs. Students particip

In [5]:
import requests

docs_url = "https://datatalks.club/faq/json/courses.json"
#print(requests.get(docs_url))
response = requests.get(docs_url)
#print(response.json())
#print (type(response.json()))
courses_raw = response.json()
#print(type(courses_raw[0]))

In [6]:
documents = []
url_prefix = "https://datatalks.club/faq"
for course in courses_raw:
    course_url = f"""{url_prefix}{course["path"]}"""
    course_response = requests.get(course_url)
    course_response.raise_for_status()
    course_data = course_response.json()
    documents.extend(course_data)
len(documents)

1208

In [7]:
documents[2]

{'id': '4d5aa45b03',
 'course': 'machine-learning-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Are Jupyter Notebooks used?',
 'answer': 'Yes. You’ll work extensively with notebooks alongside standard Python files and CLI tools.'}

In [8]:
documents[1100]

{'id': '841966c903',
 'course': 'mlops-zoomcamp',
 'section': 'Module 3: Orchestration',
 'question': 'Where is the FAQ for Prefect questions?',
 'answer': '[Here](https://docs.google.com/document/d/1Nyktf7WoRec5lDUBREXL5zLI1Edbw9_R8e45fDn4KB8/edit?usp=sharing)'}

In [9]:
from minsearch import Index

index = Index(
    text_fields = ["question", "section", "answer"],
    keyword_fields = ["course"]
)

index.fit(documents)

In [17]:
question = "I just discovered the course. Can I join now?"
#question = "When baby will born, will it be a girl or a boy"

search_results = index.search(
    question,
    boost_dict = {"question":2.0, "section":0.5},
    filter_dict = {"course": "llm-zoomcamp"},
    num_results = 5
)

search_results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '977bf7786c',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
  'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."},
 {'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you c

In [11]:
[doc["question"] for doc in search_results]

['When will the course be offered next?',
 'When and how will we be assigned projects for review/grading?',
 'OpenAI: How much will I have to spend to use the Open AI API?',
 'How is my capstone project going to be evaluated?',
 'Is it a group project?']

In [18]:
def search(question,course="llm-zoomcamp"):
    boost_dict = {"question":2.0,"section":0.5}
    filter_dict = {"course":course}

    return index.search(
        question,
        boost_dict = boost_dict,
        filter_dict = filter_dict,
        num_results=5
    )

search_results = search(question)
#search_results

In [ ]:
INSTRUCTIONS = """
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
"""
USER_PROMPT_TEMPLATE = """
Question:
{question}

Context:
{context}
"""

def build_context(search_results):
    lines = []
    for doc in search_results:
        lines.append(doc["section"])
        lines.append("Q. "+doc["question"])
        lines.append("A. "+doc["answer"])
        lines.append("")
    
    return "\n".join(lines).strip()

def build_prompt(question, search_results):
    context = build_context(search_results)
    prompt = USER_PROMPT_TEMPLATE.format(
        question = question,
        context = context
    )
    return prompt.strip()

prompt = build_prompt(question,search_results)
print(prompt)
    

Question:
I just discovered the course. Can I join now?

Context:
General Course-Related Questions
Q. I just discovered the course. Can I still join?
A. Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

General Course-Related Questions
Q. Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
A. You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

General Course-Related Questions
Q. Certificate: Can I follow the course in a self-paced mode and get a certificate?
A. No, you can only get a certificate if you finish the course with a "live" cohort.

We don't award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project

In [21]:
from pprint import pprint

#print(prompt)

response = open_client.responses.create(
     #model="google/gemini-2.5-flash",
        model="google/gemma-4-31b-it:free",
        input=INSTRUCTIONS+prompt,
        #max_output_tokens=128    
)
print(response.output_text)
#print(response.id)
#print(response.text)
#pprint(response.model_dump())
#help(response)
#print(dir(response))
#response.output[0].content[0].text
print(response.usage)

Yes, you can still join. However, if you want to receive a certificate, you need to submit your project while submissions are still being accepted.
ResponseUsage(input_tokens=422, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=31, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=453, cost=0, is_byok=False, cost_details={'upstream_inference_cost': 0, 'upstream_inference_input_cost': 0, 'upstream_inference_output_cost': 0})
